# 04_graphgrued_graph_impute_ordered5_fixed

The **complete** version for the ordered-5 branch. This notebook defaults to using the relabeled ordered-5 table and the `risk_label_5_idx` target.

- Performs an **8:1:1 stratified** split;
- Fits preprocessing **only on the training set**;
- Retains missingness in the weekly features to perform **imputation according to the protocol**;
- Benchmarks imputation strategies and then writes to the ordered-5 table.

In [6]:
# =========================
# 0) CONFIG
# =========================
BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
SPLIT_INDEX_TABLE = "dbo.tbl_graphgrued_split_index_ordered5"
IMPUTED_ALL_TABLE = "dbo.tbl_graphgrued_imputed_all_ordered5"
IMPUTER_HISTORY_TABLE = "dbo.tbl_graphgrued_imputer_history_ordered5"
FEATURE_META_TABLE = "dbo.tbl_graphgrued_feature_meta_ordered5"

# Dùng nhãn ordered-5 thật, KHÔNG scale, KHÔNG impute.
TARGET_COL = "risk_label_5_idx"

ID_COLS = ["user_id", "course_id", "enroll_time"]
LABEL_COLS = [
    "is_completed",
    "completion_rate",
    "risk_label_5_surrogate",
    "risk_label_5_idx",
    "label_d_flag",
    "label_e_flag",
    "risk_label_5_name_a",
    "risk_label_5_name_b",
    "risk_label_5_name_c",
    "risk_label_5_name_d",
    "risk_label_5_name_e",
    "course_norm_score",
    "course_raw_score_sum",
    "problems_passed",
    "total_problems",
]
GROUP_COLS = ["course_id", "school_id"]

GRAPH_HIDDEN = 128
GRAPH_DROPOUT = 0.3
GRAPH_LR = 0.01
GRAPH_WD = 5e-5
GRAPH_EPOCHS = 100
GRAPH_PATIENCE = 10
GRAPH_TAU = 0.55
GRAPH_TOP_K = 15
RUN_KNN = True
RUN_GRAPH_ONLY = True
RUN_GRAPH_ADV = True

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 8, Finished, Available, Finished, False)

In [11]:
%run utilities

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 13, Finished, Available, Finished, True)

In [12]:
# =========================
# 1) IMPORTS + LOAD HELPER MODULE
# =========================
import os
import importlib.util
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 14, Finished, Available, Finished, False)

In [13]:

# =========================
# 2) LOAD BASE TABLE
# =========================
df_base = spark.table(BASE_TABLE)
print("Rows in base table:", df_base.count())
display(df_base.limit(10))

pdf = df_base.toPandas()
print("Pandas shape:", pdf.shape)


StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 15, Finished, Available, Finished, False)

Rows in base table: 100409


SynapseWidget(Synapse.DataFrame, abe3fdf0-291c-42e9-a749-baac1e398a6b)

Pandas shape: (100409, 71)


In [14]:
# =========================
# 3) CLEAN TARGET + PICK FEATURE COLUMNS
# =========================
if TARGET_COL not in pdf.columns:
    raise ValueError(f"TARGET_COL={TARGET_COL} not found")

pdf = pdf[pdf[TARGET_COL].notna()].copy()
pdf[TARGET_COL] = pd.to_numeric(pdf[TARGET_COL], errors="coerce")
pdf = pdf[pdf[TARGET_COL].notna()].copy()
pdf[TARGET_COL] = pdf[TARGET_COL].astype(int)

valid_targets = {0, 1, 2, 3, 4}
bad_targets = sorted(set(pdf[TARGET_COL].unique().tolist()) - valid_targets)
if bad_targets:
    raise ValueError(f"Unexpected ordered-5 labels in {TARGET_COL}: {bad_targets}")

exclude = set(ID_COLS + LABEL_COLS)
feature_cols = [c for c in pdf.columns if c not in exclude]

categorical_cols = [c for c in feature_cols if str(pdf[c].dtype) == "object"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

allowed_categorical = []
for c in categorical_cols:
    nunique = pdf[c].fillna("__MISSING__").astype(str).nunique()
    if nunique <= 500:
        allowed_categorical.append(c)

categorical_cols = allowed_categorical
numeric_cols = [c for c in numeric_cols if c in feature_cols]

print("Target col       :", TARGET_COL)
print("Target values    :", sorted(pdf[TARGET_COL].dropna().unique().tolist()))
print("Numeric features :", len(numeric_cols))
print("Categorical feats:", len(categorical_cols))
print("Sample numeric   :", numeric_cols[:20])
print("Sample categorical:", categorical_cols[:20])

leak_cols = [c for c in LABEL_COLS if c in numeric_cols or c in categorical_cols]
print("Leak columns that survived feature selection:", leak_cols)
assert len(leak_cols) == 0, leak_cols

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 16, Finished, Available, Finished, False)

Target col       : risk_label_5_idx
Target values    : [0, 1, 2, 3, 4]
Numeric features : 54
Categorical feats: 3
Sample numeric   : ['year_of_birth', 'num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_watch_time_week1', 'video_watch_time_week2', 'video_watch_time_week3', 'video_watch_time_week4', 'video_count_week1', 'video_count_week2', 'video_count_week3', 'video_count_week4', 'rewind_count_week1', 'rewind_count_week2', 'rewind_count_week3', 'rewind_count_week4', 'rewind_distance_week1']
Sample categorical: ['gender', 'prerequisites', 'risk_label_5_name']
Leak columns that survived feature selection: []


In [15]:

# =========================
# 4) STRATIFIED 8:1:1 SPLIT
# =========================
split_pack = stratified_811_split(pdf, label_col=TARGET_COL, seed=SEED)

pdf["split_name"] = None
pdf.loc[split_pack.train_idx, "split_name"] = "train"
pdf.loc[split_pack.valid_idx, "split_name"] = "valid"
pdf.loc[split_pack.test_idx, "split_name"] = "test"

split_pd = pdf[["user_id", "course_id", TARGET_COL, "split_name"]].copy()
split_spark = spark.createDataFrame(split_pd)
spark.sql(f"DROP TABLE IF EXISTS {SPLIT_INDEX_TABLE}")
split_spark.write.format("delta").mode("overwrite").saveAsTable(SPLIT_INDEX_TABLE)

train_pd = pdf[pdf["split_name"] == "train"].copy()
valid_pd = pdf[pdf["split_name"] == "valid"].copy()
test_pd  = pdf[pdf["split_name"] == "test"].copy()

print(train_pd.shape, valid_pd.shape, test_pd.shape)


StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 17, Finished, Available, Finished, False)

(80327, 72) (10041, 72) (10041, 72)


In [16]:

# =========================
# 5) FIT PREPROCESSOR ON TRAIN ONLY
# =========================
pre = MixedTabularPreprocessor(numeric_cols=numeric_cols, categorical_cols=categorical_cols).fit(train_pd)
X_train_nan, M_train, feat_names = pre.transform(train_pd)
print("Transformed feature dim:", len(feat_names))

meta_rows = []
for f in feat_names:
    if f in numeric_cols:
        meta_rows.append((f, "numeric"))
    elif "__" in f:
        meta_rows.append((f, "onehot"))
    else:
        meta_rows.append((f, "derived"))
meta_df = spark.createDataFrame(meta_rows, ["feature_name", "feature_type"])
spark.sql(f"DROP TABLE IF EXISTS {FEATURE_META_TABLE}")
meta_df.write.format("delta").mode("overwrite").saveAsTable(FEATURE_META_TABLE)

print("Saved feature metadata to:", FEATURE_META_TABLE)


StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 18, Finished, Available, Finished, False)

Transformed feature dim: 462
Saved feature metadata to: dbo.tbl_graphgrued_feature_meta_ordered5


In [17]:

# =========================
# 6) BASELINE IMPUTERS
# =========================
base_outputs = []

for split_name, df_part in [("train", train_pd), ("valid", valid_pd), ("test", test_pd)]:
    X_nan, _, _ = pre.transform(df_part)
    base_outputs.append({
        "split_name": split_name,
        "zero_raw": pd.DataFrame(np.nan_to_num(X_nan, nan=0.0), columns=feat_names, index=df_part.index),
        "mean": apply_baseline_imputer(fit_baseline_imputer(pre, train_pd, "mean"), pre, df_part),
        "median": apply_baseline_imputer(fit_baseline_imputer(pre, train_pd, "median"), pre, df_part),
    })

if RUN_KNN:
    knn_imp = fit_baseline_imputer(pre, train_pd, "knn")
    for row in base_outputs:
        df_part = {"train": train_pd, "valid": valid_pd, "test": test_pd}[row["split_name"]]
        row["knn"] = apply_baseline_imputer(knn_imp, pre, df_part)


StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 19, Finished, Available, Finished, False)

In [18]:

# =========================
# 7) TRAIN GRAPH-ONLY / GRAPH-ADV IMPUTERS ON TRAIN
# =========================
train_meta = train_pd[[c for c in GROUP_COLS if c in train_pd.columns]].copy()

histories = []
graph_only_result = None
graph_adv_result = None

if RUN_GRAPH_ONLY:
    graph_only_result = train_graph_imputer(
        train_meta=train_meta,
        train_df=train_pd[numeric_cols + categorical_cols].copy(),
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        group_cols=[c for c in GROUP_COLS if c in train_pd.columns],
        hidden_dim=GRAPH_HIDDEN,
        dropout=GRAPH_DROPOUT,
        lr=GRAPH_LR,
        weight_decay=GRAPH_WD,
        epochs=GRAPH_EPOCHS,
        patience=GRAPH_PATIENCE,
        tau=GRAPH_TAU,
        top_k=GRAPH_TOP_K,
        use_adversarial=False,
    )
    h = graph_only_result.history.copy()
    h["imputer_name"] = "graph_only"
    histories.append(h)

if RUN_GRAPH_ADV:
    graph_adv_result = train_graph_imputer(
        train_meta=train_meta,
        train_df=train_pd[numeric_cols + categorical_cols].copy(),
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        group_cols=[c for c in GROUP_COLS if c in train_pd.columns],
        hidden_dim=GRAPH_HIDDEN,
        dropout=GRAPH_DROPOUT,
        lr=GRAPH_LR,
        weight_decay=GRAPH_WD,
        epochs=GRAPH_EPOCHS,
        patience=GRAPH_PATIENCE,
        tau=GRAPH_TAU,
        top_k=GRAPH_TOP_K,
        use_adversarial=True,
    )
    h = graph_adv_result.history.copy()
    h["imputer_name"] = "graph_adv"
    histories.append(h)

if histories:
    hist_pd = pd.concat(histories, ignore_index=True)
    hist_spark = spark.createDataFrame(hist_pd)
    spark.sql(f"DROP TABLE IF EXISTS {IMPUTER_HISTORY_TABLE}")
    hist_spark.write.format("delta").mode("overwrite").saveAsTable(IMPUTER_HISTORY_TABLE)
    print("Saved imputer history to:", IMPUTER_HISTORY_TABLE)


StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 20, Finished, Available, Finished, False)

Saved imputer history to: dbo.tbl_graphgrued_imputer_history_ordered5


In [19]:
# =========================
# 8) APPLY IMPUTERS TO TRAIN / VALID / TEST
# =========================
rows = []

KEEP_LABEL_BACK = [
    "risk_label_5_idx",
    "risk_label_5_surrogate",
    "label_d_flag",
    "label_e_flag",
    "is_completed",
    "completion_rate",
]

def attach_back(df_part, feature_frame, imputer_name, split_name):
    back_cols = list(dict.fromkeys(ID_COLS + [c for c in KEEP_LABEL_BACK if c in df_part.columns]))
    out = pd.concat([
        df_part[back_cols].reset_index(drop=True),
        feature_frame.reset_index(drop=True)
    ], axis=1)
    out["imputer_name"] = imputer_name
    out["split_name"] = split_name
    return out

for split_name, df_part in [("train", train_pd), ("valid", valid_pd), ("test", test_pd)]:
    base_row = [r for r in base_outputs if r["split_name"] == split_name][0]
    rows.append(attach_back(df_part, base_row["zero_raw"], "zero_raw", split_name))
    rows.append(attach_back(df_part, base_row["mean"], "mean", split_name))
    rows.append(attach_back(df_part, base_row["median"], "median", split_name))
    if RUN_KNN:
        rows.append(attach_back(df_part, base_row["knn"], "knn", split_name))

    meta_df_part = df_part[[c for c in GROUP_COLS if c in df_part.columns]].copy()
    model_input_df = df_part[numeric_cols + categorical_cols].copy()

    if graph_only_result is not None:
        graph_only_imp = apply_graph_imputer(
            fit_result=graph_only_result,
            meta_df=meta_df_part,
            data_df=model_input_df,
            group_cols=[c for c in GROUP_COLS if c in df_part.columns],
            hidden_dim=GRAPH_HIDDEN,
            dropout=GRAPH_DROPOUT,
        )
        rows.append(attach_back(df_part, graph_only_imp, "graph_only", split_name))

    if graph_adv_result is not None:
        graph_adv_imp = apply_graph_imputer(
            fit_result=graph_adv_result,
            meta_df=meta_df_part,
            data_df=model_input_df,
            group_cols=[c for c in GROUP_COLS if c in df_part.columns],
            hidden_dim=GRAPH_HIDDEN,
            dropout=GRAPH_DROPOUT,
        )
        rows.append(attach_back(df_part, graph_adv_imp, "graph_adv", split_name))

imputed_pd = pd.concat(rows, ignore_index=True)
print(imputed_pd.shape)
print("target unique by imputer:")
print(imputed_pd.groupby("imputer_name")["risk_label_5_idx"].unique().head())

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 21, Finished, Available, Finished, False)

(602454, 487)
target unique by imputer:
imputer_name
graph_adv     [3, 4, 1, 0, 2]
graph_only    [3, 4, 1, 0, 2]
knn           [3, 4, 1, 0, 2]
mean          [3, 4, 1, 0, 2]
median        [3, 4, 1, 0, 2]
Name: risk_label_5_idx, dtype: object


In [22]:

# =========================
# 9) SAVE IMPUTED DATASETS TO DELTA
# =========================
# Spark không thích tên cột có khoảng trắng hoặc ký tự lạ.
clean_cols = []
for c in imputed_pd.columns:
    cc = str(c).replace(" ", "_").replace("-", "_").replace("/", "_")
    clean_cols.append(cc)
imputed_pd.columns = clean_cols
display(imputed_pd)

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef407882-c35e-4ce9-8f0b-e5b087b461cb)

In [25]:
import re
from collections import defaultdict, Counter

def spark_safe_unique_columns(cols):
    out = []
    seen = defaultdict(int)

    for c in cols:
        # canonicalize cho Spark
        base = str(c).strip()
        base = re.sub(r"[^0-9a-zA-Z_]", "_", base)
        base = re.sub(r"_+", "_", base).strip("_").lower()
        if base == "":
            base = "col"

        k = seen[base]
        new_name = base if k == 0 else f"{base}__dup{k}"
        seen[base] += 1
        out.append(new_name)

    return out

# debug xem đang đụng cột nào
raw_cols = list(imputed_pd.columns)
canon_cols = [
    re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z_]", "_", str(c))).strip("_").lower()
    for c in raw_cols
]
dups = [k for k, v in Counter(canon_cols).items() if v > 1]

if dups:
    print("Duplicate-after-canonicalization columns found:")
    for d in dups[:50]:
        collided = [c for c in raw_cols if re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z_]", "_", str(c))).strip("_").lower() == d]
        print(f"{d} <- {collided}")

# rename an toàn
imputed_pd = imputed_pd.copy()
imputed_pd.columns = spark_safe_unique_columns(imputed_pd.columns)

# save
imputed_spark = spark.createDataFrame(imputed_pd)
spark.sql(f"DROP TABLE IF EXISTS {IMPUTED_ALL_TABLE}")
imputed_spark.write.format("delta").mode("overwrite").saveAsTable(IMPUTED_ALL_TABLE)

print(f"Saved imputed datasets to: {IMPUTED_ALL_TABLE}")
display(spark.table(IMPUTED_ALL_TABLE).limit(20))

StatementMeta(, 7b8e418e-36b2-45f4-8a55-85164a99f8a1, 27, Finished, Available, Finished, False)

Duplicate-after-canonicalization columns found:
prerequisites <- ['prerequisites__', 'prerequisites_________作为学前教育、早期教育、幼儿发展与健康管理专业的学生，要完成整个知识和能力体系的专业构建，需_要相关专业课程的配套组合。这三个专业学生的前承课程均为生理卫生学和解剖学。', 'prerequisites________《民族居住环境与传续设计》是云南艺术学院设计学院环境设计专业的特色课程，强调理论、知识与具体设计实践的结合。修学该课程必需具备全面的环境设计综合知识，并有良好的实训功底。', 'prerequisites______本课程的先修课程包括：高等数学、线性代数、电路原理、工程电磁场、电机学。', 'prerequisites___具备一定的系统解剖学和局部解剖学基础知识。', 'prerequisites__Нет', 'prerequisites__《中医基础理论》《中医诊断学》《经络腧穴学》《刺法灸法学》《针灸治疗学》等基础课程。', 'prerequisites__《古代汉语》和《中国古代文学》', 'prerequisites__《地球科学概论》、《矿物岩石学》、《构造地质学》、《地貌第四纪地质》', 'prerequisites__《基础会计》、《政府预算》', 'prerequisites__《基础素描》、《透视》', 'prerequisites__《学前儿童卫生与保健》研究学前儿童生理发展的特点和规律，为学生学习学前儿童的心理发展特点提供生理基础。', 'prerequisites__《微积分》或《高等数学》', 'prerequisites__《旅游学概论》、《旅游资源学》、《市场营销学》、《旅游经济学》、《旅游消费者行为学》、《旅行社经营与管理》。', 'prerequisites__《无机化学》、《有机化学》、《物理化学》、《分析化学》、《化工原理》和《植物纤维化学》等课程相关知识。', 'prerequisites__《机械制图》', 'prerequisites__《材料力学》、《结构力学》、《高等数学》、《建筑材料》、《房屋建筑学》', 'prerequisites__《消防识图与制图》《固定消防设施》', 'pre

SynapseWidget(Synapse.DataFrame, ebaf9cac-d09f-4968-aa2d-0b4c4792f49b)